In [ ]:
"""Locate one hash-bound input across Kaggle's supported mount layouts."""
import hashlib
from pathlib import Path


def locate_input(root, manifest_sha256):
    root = Path(root).resolve()
    matches = []
    candidates = list(root.rglob("INPUT_MANIFEST.json"))
    if len(candidates) > 20:
        raise RuntimeError("Too many candidate manifests; require explicit input review")
    for candidate in candidates:
        candidate = candidate.resolve()
        if not candidate.is_relative_to(root):
            raise ValueError("Input manifest resolves outside Kaggle input mount")
        if hashlib.sha256(candidate.read_bytes()).hexdigest() == manifest_sha256:
            matches.append(candidate.parent)
    if len(matches) != 1:
        names = [str(path.relative_to(root)) for path in candidates]
        raise RuntimeError(f"Expected exactly one hash-bound input, found {len(matches)}; candidates={names}")
    return matches[0]

import sys
input_root=locate_input('/kaggle/input','bf4410686ed3efe8d7df4fa346bf957b8cd61adfac0df2facda32f12e67c4001')
sys.path.insert(0,str(input_root))

import compiler_probe_job
original_install=compiler_probe_job.install
def install_with_executable_modes(input_root,destination):
    python,environment=original_install(input_root,destination)
    for simulator in (destination/'site').glob('nncase.simulator.*.sc'):
        simulator.chmod(0o755)
    return python,environment
compiler_probe_job.install=install_with_executable_modes
compiler_probe_job.run(input_root,'k230-topface-v11-compiler-linux-a02')
